# What an Encoding Knows — a runnable companion

This notebook is a cell-by-cell companion to the paper *"What an Encoding Knows: An
Exploration of Invariants Witnessed by Compressed Column Layouts"*.

**How to read it.** Every code cell begins with a comment naming the exact place in
the paper it corresponds to, like `# §2 ¶3`. Open the paper beside the notebook and
read them in parallel. Running every cell top to bottom reproduces the paper's
argument: the toy examples are computed live, and the measured results are loaded
from the artifact's own committed CSVs.

**What is real and what is illustrative.**

| Source | Used for |
|---|---|
| `experiments/results/**.csv` | every measured number; these are the artifact's canonical evidence |
| `experiments/results/claim_manifest.csv` | the authoritative claim-name → value mapping the paper renders |
| synthetic values defined in-cell | the small pedagogical examples (§2, §3), clearly marked `DUMMY` |

If the CSVs are absent (for example if you are reading this outside the repository)
every cell still runs: the loader falls back to a small synthetic stand-in and says
so loudly. Numbers in that mode are **not** the paper's numbers.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SETUP — no paper section. Imports, palette, and the data loader.
# ═════════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import csv
import math
import os
import textwrap
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# The paper's categorical slots 1-2. Validated for colour-vision deficiency:
# normal-vision dE 33.6, deuteranopia 31.6, protanopia 24.5 (OKLab x100).
# Well above the >=8 CVD target, so the two series stay distinct for every reader.
BLUE, ORANGE = "#2A78D6", "#EB6834"
INK, MUTED, GRID = "#18181B", "#52525B", "#D4D4D8"

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.5, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "legend.frameon": False,
})

# Locate the repository root whether the notebook sits at the root or in notebooks/.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "experiments" / "results").is_dir():
        ROOT = candidate
        break
RESULTS = ROOT / "experiments" / "results"
LIVE = RESULTS.is_dir()

print(f"repository root : {ROOT}")
print(f"measured data   : {'FOUND — numbers below are the artifact’s own' if LIVE else 'ABSENT — synthetic fallback, numbers are ILLUSTRATIVE ONLY'}")


def load_csv(relative, fallback_rows=None):
    """Read a canonical CSV as a list of dicts, or fall back to a synthetic stand-in."""
    path = RESULTS / relative
    if path.is_file():
        with path.open(newline="") as handle:
            return list(csv.DictReader(handle))
    if fallback_rows is None:
        raise FileNotFoundError(f"{relative} missing and no fallback supplied")
    print(f"  !! {relative} absent — using DUMMY rows; not the paper's numbers")
    return fallback_rows


def manifest():
    """The claim-name -> displayed-value mapping the paper's macros are generated from."""
    path = RESULTS / "claim_manifest.csv"
    if not path.is_file():
        print("  !! claim_manifest.csv absent — claim lookups will show <dummy>")
        return {}
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        next(reader)
        return {row[0]: row[1] for row in reader if len(row) >= 2}


CLAIMS = manifest()


def claim(name):
    """Look up exactly what the paper prints for a claim macro."""
    return CLAIMS.get(name, "<dummy>")


def rule(title=""):
    print("\n" + "─" * 78)
    if title:
        print(title)
        print("─" * 78)


def wrap(text, indent=""):
    print(textwrap.fill(" ".join(text.split()), 78,
                        initial_indent=indent, subsequent_indent=indent))


print(f"claims loaded   : {len(CLAIMS)}")


# ── the bridge onto the REAL Rust pipeline ──────────────────────────────────
# Everything in §2 and §3 below drives the actual encoder, invariant calculus
# and compiler through this, rather than reimplementing them in Python. If the
# binary is missing, build it once:
#     cargo build --release --features experiment --bin witness_explore
import json as _json
import subprocess

EXPLORE = ROOT / "target" / "release" / "witness_explore"
BRIDGE = EXPLORE.is_file()


def witness(subcommand, **payload):
    """Call the real pipeline and return its JSON reply."""
    if not BRIDGE:
        raise RuntimeError(
            "witness_explore not built. Run:\n"
            "  cargo build --release --features experiment --bin witness_explore"
        )
    result = subprocess.run(
        [str(EXPLORE), subcommand, _json.dumps(payload)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "witness_explore failed")
    return _json.loads(result.stdout)


print(f"rust bridge     : {'READY — §2/§3 drive the real compiler' if BRIDGE else 'MISSING (build it; see comment above)'}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# ABSTRACT — the four numbers the abstract puts on the table, straight from
# the manifest. Every one of these is regenerated from CSV; none is typed in.
# ═════════════════════════════════════════════════════════════════════════════
rule("ABSTRACT — the claims, as the paper prints them")

print(f"""
  Census scope        {claim('WitCensusColumns')} columns, {claim('WitCensusSources')} sources, {claim('WitCensusRowsMillions')}M rows
  Global monotonicity {claim('WitCensusMonotoneColumns')}% of columns
  Query cells         {claim('WitCells')} range-and-aggregate queries over {claim('WitPairs')} pairs

  Discovery share     median {claim('WitDiscoveryShare')} of complete query time
  Access-ready vs size-selected
                      {claim('WitSourceDirectStorageMedian')}x the time, at {claim('WitAccessPremium')}x the bytes
  Access-ready vs sortedness-aware Parquet
                      {claim('WitSourceDirectBoundaryMedian')}x over source medians
                      95% bootstrap CI [{claim('WitSourceDirectBoundaryCiLow')}, {claim('WitSourceDirectBoundaryCiHigh')}]
                      cluster bootstrap  [{claim('WitClusterDirectBoundaryCiLow')}, {claim('WitClusterDirectBoundaryCiHigh')}]
""")

wrap("""The thesis in one line: an encoding does not merely store values compactly, it
constrains them — and a query planner that can name the constraint may run a
different algorithm on the same bytes. The rest of this notebook builds that claim
from the ground up, then shows what it is worth on real columns.""")

---
## §1 Introduction

The paper opens with a query and a question: what does the *representation* prove
about its values, and which bytes must be readable to exploit that proof?

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1 ¶1 — The motivating query.
# DUMMY DATA: the paper's illustrative table, defined here so you can run it.
# ═════════════════════════════════════════════════════════════════════════════
time    = [1000, 1003, 1004, 1004, 1007, 1012, 1012, 1015, 1101, 1101, 1104, 1110]
reading = [  10,   11,   12,   13,   14,   15,   16,   17,   18,   19,   20,   21]

LOW, HIGH = 1004, 1015

rule("§1 — SUM(reading) WHERE 1004 <= time <= 1015")

# The only strategy available to a reader that can decode and nothing else.
matching = [i for i, t in enumerate(time) if LOW <= t <= HIGH]
print(f"  time      : {time}")
print(f"  matching rows : {matching}   -> SUM(reading) = {sum(reading[i] for i in matching)}")
print(f"  values examined by a plain scan: {len(time)} of {len(time)}")

wrap("""A reader that knows only how to decode `time` has exactly one safe strategy:
decode every value, compare it, keep the row positions that match. That is correct,
and for an unsorted column it is also the best available. The question the paper
asks is whether the bytes themselves ever license something better.""")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1 ¶2 — The two questions the paper separates.
# ═════════════════════════════════════════════════════════════════════════════
rule("§1 — two questions storage systems usually mix")

questions = [
    ("What does the representation PROVE about its values?",
     "a fact: e.g. 'these values never decrease'",
     "authorises an algorithm (binary search instead of scan)"),
    ("Which BYTES must be available to exploit that proof?",
     "an access capability: e.g. 'a probe can land on a chosen row'",
     "decides whether the algorithm is physically worth running"),
]
for i, (q, kind, effect) in enumerate(questions, 1):
    print(f"\n  {i}. {q}")
    print(f"       is        : {kind}")
    print(f"       determines: {effect}")

rule()
wrap("""Neither implies the other, and that independence is the whole point. A sorted
column sealed inside one opaque compressed frame still proves order — but reaching
row 5000 may require decompressing the frame, so the proof buys nothing. Restart
anchors make probing cheap but say nothing whatever about order. A useful
optimisation needs both, and the paper's contribution is a contract that keeps them
separate so a planner can check for each independently.""")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1 ¶4 — The three pieces Witness provides.
# ═════════════════════════════════════════════════════════════════════════════
rule("§1 — the three pieces")

pieces = {
    "A representation contract":
        "decoder semantics say how values are reconstructed; layout rules say where "
        "bytes live and what must be read with them; typed facts say what is known, "
        "why, and under which assumptions.",
    "A fact-gated compiler":
        "rules derive facts bottom-up through composed codecs. Every plan step that "
        "avoids a scan must cite a sufficient fact; a checker re-verifies the "
        "citation before the plan runs; invalid evidence yields an explicit scan.",
    "An empirical study":
        "Q1 availability — how often do real columns provide useful facts? "
        "Q2 effect — which part of a query does a fact remove? "
        "Q3 price — how many bytes keep that fact usable?",
}
for name, body in pieces.items():
    print(f"\n  {name}")
    wrap(body, indent="      ")

rule()
wrap("""Note what is deliberately NOT claimed: Witness is not a universal codec and does
not claim to beat Parquet. The contribution is a disciplined way to derive, consume
and measure what an encoded layout already implies.""")

---
## §2 One Query, Two Legal Plans

Every cell below calls the **real** encoder, invariant calculus and compiler through
`witness_explore`. Nothing here is a Python re-implementation — what you see printed
is what the compiler actually decided.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶1 — The selector column. DUMMY VALUES (the paper's twelve), but from here
# on every derivation is performed by the Rust crate, not by this notebook.
# ═════════════════════════════════════════════════════════════════════════════
X = [1000, 1003, 1004, 1004, 1007, 1012, 1012, 1015, 1101, 1101, 1104, 1110]
LOW, HIGH = 1004, 1015

rule("§2 — the column, and what the real encoder makes of it")
print(f"  values : {X}")
print(f"  query  : FILTER {LOW} <= x <= {HIGH}     (expected rows [2, 8))")

page = witness("encode", values=X, recipe="UnsignedDelta(1024, BitPack)")
print(f"\n  encoded page      : {page['page_bytes']} bytes, {page['rows']} rows, "
      f"{len(page['fields'])} fields, {page['frames']} frames")
print(f"  descriptor says   : non_decreasing={page['checked_non_decreasing']}, "
      f"nulls={page['checked_null_placement']}")
print(f"\n  {'field':10s} {'bytes':>7s} {'align':>6s} {'granularity':>12s} {'location':>9s} {'offset':>7s}")
for f in page["fields"]:
    print(f"  {f['name']:10s} {f['length']:>7} {f['alignment']:>6} "
          f"{f['read_granularity']:>12} {f['location']:>9} {f['offset']:>7}")

wrap("""These are real serialized bytes with real offsets — the same ACPAGE01 format the
study measures. The metadata field spans the aligned header; the rest are the streams
the decoder will read.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶3 — Signed vs unsigned coding. THE REAL CALCULUS, and it is subtler than
# "signed means no fact". Watch the EVIDENCE column, not the property column.
# ═════════════════════════════════════════════════════════════════════════════
rule("§2 — same values, two codings, as derived by the real invariant calculus")

for recipe in ["UnsignedDelta(1024, BitPack)", "Delta(1024, BitPack)"]:
    got = witness("facts", values=X, recipe=recipe)
    print(f"\n  {recipe}   ({got['fact_count']} facts)")
    for f in got["facts"]:
        if "NonDecreasing" in f["property"]:
            print(f"      {f['property']:34s} evidence = {f['evidence']}")

rule()
wrap("""This is the correction the real system forces. Both codings yield
Value(NonDecreasing) for THIS column — but for different reasons. Unsigned delta gets
Structural(\"unsigned_delta_segments\"): the encoding cannot represent a decrease, so
the fact is free and cannot go stale. Signed delta gets
CheckedDescriptor(\"non_decreasing\"): the encoder inspected the values, found them
ordered, and persisted an authenticated claim.""", indent="  ")
wrap("""Same property, same plan, different trust. The structural fact needs no faith in
the encoder; the checked one does. §3.3 is where that distinction is cashed out.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶3b — So where does the fact actually DISAPPEAR? Give the encoder a column
# that genuinely is not ordered.
# ═════════════════════════════════════════════════════════════════════════════
UNORDERED = [1000, 1003, 900, 1004, 1007, 1012, 880, 1015, 1101, 1101, 1104, 1110]

rule("§2 — a column with real decreases")
print(f"  values : {UNORDERED}")
print(f"  descending steps present: "
      f"{sum(1 for a, b in zip(UNORDERED, UNORDERED[1:]) if b < a)}")

got = witness("facts", values=UNORDERED, recipe="Delta(1024, BitPack)")
order_facts = [f for f in got["facts"] if "NonDecreasing" in f["property"]]
print(f"\n  facts derived : {got['fact_count']}")
print(f"  order facts   : {len(order_facts)}  <- none; there is nothing true to certify")
for f in got["facts"]:
    print(f"      {f['scope']:9s} {f['property'][:40]:40s} {f['evidence']}")

wrap("""The encoder cannot certify what is false, and unsigned coding cannot even
represent this column's steps. So no order fact exists under any evidence class. This
is the honest boundary of the mechanism — and the next cell shows the compiler
respecting it.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶5 — The two legal plans, compiled by the REAL compiler. Same query,
# different columns, different authorised algorithms.
# ═════════════════════════════════════════════════════════════════════════════
rule("§2 — what the compiler emits")

QUERY = {"kind": "filter_between", "low": LOW, "high": HIGH}

for label, values in [("ordered column", X), ("unordered column", UNORDERED)]:
    plan = witness("compile", values=values, recipe="Delta(1024, BitPack)", query=QUERY)
    print(f"\n  {label}   ->  output guarantee {plan['output']}")
    for n in plan["nodes"]:
        cites = n["cites"]
        marker = "  <- FAST PATH" if cites != "unconditional" else ""
        print(f"      {n['op'][:44]:44s} cites={cites}{marker}")

rule()
wrap("""The ordered column compiles to SearchMonotone, and the node CITES the fact that
makes it legal. The unordered column gets SeekRestart + ReadRange + RefineCandidates —
a scan — and no node claims any fact. That citation is not decoration: a checker
re-verifies it before the plan runs, so a step that skips scan work must name a fact
the column actually proves.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶5b — Execute both, through the real runtime. Same answer, real byte
# accounting from ReadSession.
# ═════════════════════════════════════════════════════════════════════════════
rule("§2 — execution and measured bytes")

print(f"  {'column':18s} {'rows returned':>14s} {'logical B':>10s} {'delivered B':>12s} {'page B':>8s}")
print(f"  {'-'*18} {'-'*14} {'-'*10} {'-'*12} {'-'*8}")
answers = {}
for label, values in [("ordered", X), ("unordered", UNORDERED)]:
    got = witness("run", values=values, recipe="Delta(1024, BitPack)", query=QUERY)
    answers[label] = got["answer"]
    print(f"  {label:18s} {got['answer'].get('rows', 0):>14} {got['logical_bytes']:>10} "
          f"{got['delivered_bytes']:>12} {got['page_bytes']:>8}")

print(f"\n  ordered   ranges: {answers['ordered'].get('ranges')}")
print(f"  unordered ranges: {answers['unordered'].get('ranges')}")

# Cross-check the real engine against a brute-force count computed here.
for label, values in [("ordered", X), ("unordered", UNORDERED)]:
    expected = sum(1 for v in values if LOW <= v <= HIGH)
    assert answers[label].get("rows") == expected, f"{label}: engine disagrees with brute force"
print(f"\n  both answers cross-checked against brute force: OK")
wrap("""Delivered bytes are what the layout actually forced the reader to fetch, counted
by the runtime rather than estimated here. This counter is what §5.4 turns into the
storage/access frontier.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶6 — The case the paper keeps returning to: a fact that is TRUE but not
# cheaply USABLE. A Zstd frame preserves the value fact and erases the seek.
# ═════════════════════════════════════════════════════════════════════════════
rule("§2 — fact present, access absent")

# A larger column so the frame effect is not lost in header noise.
BIG = list(range(0, 8192, 2))
BIGQ = {"kind": "filter_between", "low": 1000, "high": 1100}

print(f"  {'recipe':36s} {'page B':>8s} {'logical B':>10s} {'deliv. B':>9s} {'frames':>7s} {'rows':>5s}")
print(f"  {'-'*36} {'-'*8} {'-'*10} {'-'*9} {'-'*7} {'-'*5}")
for recipe in ["UnsignedDelta(256, BitPack)",
               "UnsignedDelta(4096, BitPack)",
               "Frame(UnsignedDelta(256, BitPack))"]:
    got = witness("run", values=BIG, recipe=recipe, query=BIGQ)
    facts = witness("facts", values=BIG, recipe=recipe)
    has_order = any("NonDecreasing" in f["property"] for f in facts["facts"])
    print(f"  {recipe:36s} {got['page_bytes']:>8} {got['logical_bytes']:>10} "
          f"{got['delivered_bytes']:>9} {got['frames_decoded']:>7} "
          f"{got['answer'].get('rows', 0):>5}")

rule()
wrap("""The framed page is the SMALLEST on disk and still carries the order fact — every
recipe here derives NonDecreasing. What separates it is `frames_decoded`: reaching any
byte inside the frame costs a whole-frame decode, so the fact cannot be spent cheaply.
That is the fact/capability split, measured rather than asserted, and it is exactly
the trade §5.4 prices across 32 real sources.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 Figure — restart interval sets the probe cost. Shorter restarts mean
# cheaper selective access and more anchor bytes: the frontier in miniature.
# Form: paired bars, one axis, two series (bytes vs delivered) -> legend + table.
# ═════════════════════════════════════════════════════════════════════════════
intervals = [128, 256, 512, 1024, 2048, 4096]   # must be miniblock (128-row) multiples
page_bytes, delivered = [], []
for r in intervals:
    got = witness("run", values=BIG, recipe=f"UnsignedDelta({r}, BitPack)", query=BIGQ)
    page_bytes.append(got["page_bytes"])
    delivered.append(got["delivered_bytes"])

fig, ax = plt.subplots(figsize=(7.6, 3.6))
idx = np.arange(len(intervals))
ax.bar(idx - 0.19, page_bytes, width=0.36, color=BLUE, label="page size (bytes stored)")
ax.bar(idx + 0.19, delivered, width=0.36, color=ORANGE, label="delivered (bytes read)")
ax.set_xticks(idx); ax.set_xticklabels(intervals)
ax.set_xlabel("delta restart interval (rows)"); ax.set_ylabel("bytes")
ax.set_title("§2 — shorter restarts buy cheaper probes with more anchor bytes", loc="left")
ax.legend(); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

print(f"  {'interval':>9s} {'page B':>8s} {'delivered B':>12s} {'delivered/page':>15s}")
for r, p, d in zip(intervals, page_bytes, delivered):
    print(f"  {r:>9} {p:>8} {d:>12} {d/p:>14.2f}x")
wrap("""This is the storage/access frontier at the scale of one column: the same trade
the study measures across the corpus, where the access-ready menu pays about twice the
bytes of the size-selected one.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2 ¶6b — A constraint the real encoder enforces, discovered by asking it.
# Restart intervals are not free parameters: they must be positive multiples of
# the 128-row miniblock, because a restart has to land on a miniblock boundary
# for a probe to be able to start decoding there at all.
# ═════════════════════════════════════════════════════════════════════════════
rule("§2 — the encoder rejects layouts it cannot honour")

print(f"  {'recipe':34s} {'accepted':>9s}  reason if rejected")
print(f"  {'-'*34} {'-'*9}  {'-'*46}")
for spec in ["UnsignedDelta(64, BitPack)", "UnsignedDelta(128, BitPack)",
             "UnsignedDelta(192, BitPack)", "UnsignedDelta(256, BitPack)",
             "Rle(4, BitPack)", "Patch(4, BitPack)"]:
    try:
        witness("facts", values=X, recipe=spec)
        print(f"  {spec:34s} {'yes':>9}")
    except RuntimeError as exc:
        print(f"  {spec:34s} {'no':>9}  {str(exc).strip(chr(34))[:46]}")

wrap("""This matters for reading §5.4. The access-ready menu \"shortens delta restart
intervals\" — but it cannot shorten them arbitrarily, and the granularity floor is why
the byte premium is lumpy across columns rather than a smooth dial. Run-length and
patch indexes have no such constraint, which is why they accept an interval of 4.""",
     indent="  ")

---
## §3 From Bytes to an Authorized Plan

Three ideas carry the paper and they are independent. Each cell below asks the real
crate rather than describing it.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3 ¶1 + Table 1 — the vocabulary, then the same three things read out of a
# real encoded page so the words attach to something concrete.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3 Table 1 — vocabulary")
for term, meaning, example in [
    ("Fact",              "known of the values",  "values never decrease"),
    ("Access capability", "cheap to reach",       "restart anchors"),
    ("Authorized plan",   "legal fast algorithm", "boundary search"),
    ("Property",          "the proposition",      "NonDecreasing"),
    ("Evidence",          "why it is believed",   "unsigned per-step delta"),
    ("Guarantee",         "what a plan returns",  "the matching rows"),
]:
    print(f"  {term:20s} {meaning:24s} {example}")

rule("the same three, taken from a real page")
facts = witness("facts", values=X, recipe="UnsignedDelta(256, BitPack)")
plan = witness("compile", values=X, recipe="UnsignedDelta(256, BitPack)",
               query={"kind": "filter_between", "low": LOW, "high": HIGH})

value_facts = [f for f in facts["facts"] if f["scope"] == "AllRows"]
access_facts = [f for f in facts["facts"] if f["scope"] == "Physical"]
fast = [n for n in plan["nodes"] if n["cites"] != "unconditional"]

print(f"  FACTS (value)      : {[f['property'] for f in value_facts]}")
print(f"  CAPABILITIES       : {[f['property'] for f in access_facts]}")
print(f"  AUTHORIZED PLAN    : {[n['op'] for n in fast]}")
print(f"    ... citing       : {[n['cites'] for n in fast]}")
wrap("""Value facts live in scope AllRows; access capabilities live in scope Physical.
The compiler needs one of each before it will emit a fast path — which is why the two
scopes exist in the type at all.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.1 — Three descriptions of one column, read out of the real serialized page.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.1 — decoder semantics / physical layout / typed facts")

page = witness("encode", values=X, recipe="Frame(UnsignedDelta(256, BitPack))")
direct = witness("encode", values=X, recipe="UnsignedDelta(256, BitPack)")

print("  (1) DECODER SEMANTICS — which values does the page represent?")
print(f"      recipe drives a decoder tree; the derived root fact is:")
for f in witness("facts", values=X, recipe="UnsignedDelta(256, BitPack)")["facts"]:
    if "NonDecreasing" in f["property"]:
        print(f"        {f['property']}  via  {f['evidence']}")

print("\n  (2) PHYSICAL LAYOUT — which bytes does a row range cost?")
print(f"      {'recipe':14s} {'fields':>7s} {'frames':>7s} {'deps':>5s} {'page B':>8s}")
print(f"      {'direct':14s} {len(direct['fields']):>7} {direct['frames']:>7} "
      f"{direct['dependencies']:>5} {direct['page_bytes']:>8}")
print(f"      {'framed':14s} {len(page['fields']):>7} {page['frames']:>7} "
      f"{page['dependencies']:>5} {page['page_bytes']:>8}")

print("\n  (3) TYPED FACTS — what is known, why, and under which assumptions?")
for f in witness("facts", values=X, recipe="UnsignedDelta(256, BitPack)")["facts"][:4]:
    print(f"        {f['scope']:9s} {f['property'][:36]:36s} {f['evidence'][:34]:34s} "
          f"{f['assumptions']}")

wrap("""Keeping these three apart prevents two errors: a decoder identity does not make
an operation cheap, and a useful layout on its own proves nothing about values. The
framed and direct pages above decode to identical values and carry identical value
facts — they differ only in description (2).""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.2 — The three fact families, enumerated from the real calculus across
# several recipes. Note which recipes contribute which scope.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.2 — value / mapping / access facts, by recipe")

RECIPES = [
    "BitPack",
    "For(BitPack)",
    "Delta(256, BitPack)",
    "UnsignedDelta(256, BitPack)",
    "Dictionary(BitPack)",
    "Rle(64, BitPack)",
    "Frame(UnsignedDelta(256, BitPack))",
]
SORTED_INPUT = [10, 10, 20, 30, 30, 30, 40, 55, 55, 70, 80, 95]

print(f"  {'recipe':36s} {'#facts':>6s}  properties")
print(f"  {'-'*36} {'-'*6}  {'-'*34}")
for recipe in RECIPES:
    try:
        got = witness("facts", values=SORTED_INPUT, recipe=recipe)
    except RuntimeError as exc:
        print(f"  {recipe:36s} {'—':>6}  rejected: {str(exc)[:40]}")
        continue
    props = sorted({f["property"].split("(")[0] + ":" + f["property"].split("(")[1].split("{")[0].split(")")[0]
                    for f in got["facts"]})
    print(f"  {recipe:36s} {got['fact_count']:>6}  {', '.join(p.strip() for p in props)[:60]}")

rule()
wrap("""Value facts (scope AllRows) constrain reconstructed values. Mapping facts describe
translation layers — a sorted deduplicated dictionary is injective and order-preserving,
so a value range rewrites exactly into a code range. Access facts (scope Physical)
describe restart, run, patch and rank indexes, and they are what determines the byte
closure.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.2 (b) — Equation (2) in the real system: a dictionary's ORDER-PRESERVING
# mapping, and the trap it does NOT license.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.2 — Eq. (2), and what it does not prove")

# Values deliberately NOT in ascending row order, but with a small domain.
SHUFFLED = [95, 10, 55, 30, 80, 10, 40, 70, 20, 30, 55, 10]
got = witness("facts", values=SHUFFLED, recipe="Dictionary(BitPack)")

mapping = [f for f in got["facts"] if f["scope"] == "Mapping"]
order   = [f for f in got["facts"] if "NonDecreasing" in f["property"]]
print(f"  values (row order) : {SHUFFLED}")
print(f"  mapping facts      : {[f['property'] for f in mapping] or 'none reported at Mapping scope'}")
print(f"  ROW-ORDER facts    : {[f['property'] for f in order] or 'NONE'}")
for f in got["facts"]:
    print(f"      {f['scope']:9s} {f['property'][:44]:44s} {f['evidence'][:30]}")

plan = witness("compile", values=SHUFFLED, recipe="Dictionary(BitPack)",
               query={"kind": "filter_between", "low": 30, "high": 70})
print(f"\n  compiled plan for FILTER 30..70:")
for n in plan["nodes"]:
    print(f"      {n['op'][:46]:46s} cites={n['cites']}")

wrap("""The dictionary mapping is order-preserving by construction, so a value range can
be rewritten into a code range with two probes and no row reads. But the ROWS are not
ordered, and the calculus does not pretend otherwise — there is no NonDecreasing fact
here, so the code stream is still scanned. Conflating 'the mapping is ordered' with
'the rows are ordered' is precisely the error the typed contract prevents.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.2 (c) — ACCESS facts decide the byte closure. Measured, not modelled:
# vary the restart interval and watch delivered bytes move.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.2 — access closure, measured through the real runtime")

RUN = list(range(0, 8192, 2))
POINT = {"kind": "get", "row": 3000}

print(f"  point lookup at row 3000")
print(f"  {'recipe':34s} {'logical B':>10s} {'delivered B':>12s} {'deliv/logical':>14s}")
print(f"  {'-'*34} {'-'*10} {'-'*12} {'-'*14}")
for recipe in ["UnsignedDelta(128, BitPack)", "UnsignedDelta(512, BitPack)",
               "UnsignedDelta(4096, BitPack)", "Frame(UnsignedDelta(128, BitPack))"]:
    got = witness("run", values=RUN, recipe=recipe, query=POINT)
    ratio = got["delivered_bytes"] / max(got["logical_bytes"], 1)
    print(f"  {recipe:34s} {got['logical_bytes']:>10} {got['delivered_bytes']:>12} {ratio:>13.2f}x")

wrap("""A delta lookup needs the preceding restart anchor and every delta between it and
the target, so a longer restart interval forces more bytes for the same single row.
The framed row shows the extreme: the smallest possible logical request still delivers
the whole frame. The closure is a least fixed point over layout prerequisites, and
these are its consequences in bytes.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.3 — Evidence classes, taken from real derivations. Same property,
# different reason, different trust boundary.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.3 — where each evidence class actually comes from")

samples = [
    ("UnsignedDelta(1024, BitPack)", X,        "unsigned coding forbids a decrease"),
    ("Delta(1024, BitPack)",         X,        "encoder inspected and certified"),
    ("BitPack",                      X,        "encoder inspected and certified"),
    ("Delta(1024, BitPack)",         UNORDERED,"nothing true to certify"),
]
print(f"  {'recipe':30s} {'column':11s} {'evidence for NonDecreasing':44s}")
print(f"  {'-'*30} {'-'*11} {'-'*44}")
for recipe, values, note in samples:
    got = witness("facts", values=values, recipe=recipe)
    ev = next((f["evidence"] for f in got["facts"] if "NonDecreasing" in f["property"]), "— none —")
    label = "ordered" if values is X else "unordered"
    print(f"  {recipe:30s} {label:11s} {ev[:44]:44s}")

rule()
print("  Structural         encoding CANNOT represent a violation; free; cannot go stale")
print(f"  CheckedDescriptor  encoder verified + persisted, authenticated; "
      f"{claim('WitCertificateHeaderBytes')} B descriptor")
print("  Layout             the serialized dependency graph entails it; free")
wrap("""The honest limitation the paper states plainly: a checksum authenticates BYTES,
not the truth of the claim they carry. A structural fact needs no faith in the encoder;
a checked one does. That is weaker than proof-carrying code, where the consumer checks
an attached proof rather than trusting the producer.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.3 (b) — Output guarantees, read off real compiled plans. The type is what
# stops a candidate set being mistaken for an answer.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.3 — guarantees the real compiler attaches")

queries = [
    ("GET row 5",          {"kind": "get", "row": 5}),
    ("SUM all rows",       {"kind": "sum"}),
    ("BETWEEN 1004..1015", {"kind": "between", "low": LOW, "high": HIGH}),
    ("FILTER 1004..1015",  {"kind": "filter_between", "low": LOW, "high": HIGH}),
    ("FILTER = 1012",      {"kind": "filter_equals", "value": 1012}),
]
print(f"  {'query':22s} {'output guarantee':22s} {'nodes':>6s}  ops")
print(f"  {'-'*22} {'-'*22} {'-'*6}  {'-'*30}")
for label, q in queries:
    try:
        plan = witness("compile", values=X, recipe="UnsignedDelta(256, BitPack)", query=q)
    except RuntimeError as exc:
        print(f"  {label:22s} {'REJECTED':22s} {'—':>6}  {str(exc)[:40]}")
        continue
    ops = ", ".join(n["op"].split(" ")[0].strip("{") for n in plan["nodes"])
    print(f"  {label:22s} {plan['output']:22s} {plan['node_count']:>6}  {ops[:44]}")

wrap("""CandidateBitmap is the load-bearing one: a Bloom hit or a min/max overlap is a
SUPERSET, and the type forces refinement before the rows become an answer. A miss, by
contrast, is exact — absence really is an answer.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.4 — Equation (3) and the checker. Every fast node names the fact that
# makes it legal, and the byte closure states what it may read.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3.4 — a compiled plan in full, from the real compiler")

plan = witness("compile", values=list(range(0, 8192, 2)),
               recipe="UnsignedDelta(256, BitPack)",
               query={"kind": "filter_between", "low": 1000, "high": 1100})

print(f"  query    : {plan['query'][:70]}")
print(f"  output   : {plan['output']}\n")
for n in plan["nodes"]:
    print(f"  node {n['id']}  {n['op'][:52]}")
    print(f"          cites     : {n['cites']}")
    print(f"          guarantee : {n['guarantee']}")
    print(f"          closure   : {n['closure']}"
          + (f" ({n['closure_bytes']} B)" if n['closure'] == 'exact' else f" — {n['closure_note'][:44]}"))

rule()
wrap("""RuntimeRefined is not vagueness — it is honesty. When IDs, ranks, runs, patches or
search probes determine later fields at run time, the compiler declares a conservative
envelope and refines it after reading an index, rather than printing a false static
byte count. The paper's reported byte numbers refer to this declared layout.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.4 (b) + Table 2 — derivation rules, verified against the real calculus,
# including the two that deliberately derive nothing useful.
# ═════════════════════════════════════════════════════════════════════════════
rule("§3 Table 2 — claimed rule vs what the crate actually derives")

ASC = [10, 10, 20, 30, 30, 30, 40, 55, 55, 70, 80, 95]
checks = [
    ("UnsignedDelta(128, BitPack)", ASC, "order within each restart span"),
    ("Dictionary(BitPack)",       ASC, "order-preserving code mapping"),
    ("Rle(4, BitPack)",           ASC, "run values inherit order"),
    ("Patch(4, BitPack)",         ASC, "NOTHING — one exception breaks order"),
    ("Frame(UnsignedDelta(128, BitPack))", ASC, "value fact kept, cheap seek erased"),
]
print(f"  {'recipe':36s} {'order fact?':>12s}  claimed behaviour")
print(f"  {'-'*36} {'-'*12}  {'-'*34}")
for recipe, values, claimed in checks:
    try:
        got = witness("facts", values=values, recipe=recipe)
        has = any("NonDecreasing" in f["property"] for f in got["facts"])
        ev = next((f["evidence"].split("(")[0] for f in got["facts"]
                   if "NonDecreasing" in f["property"]), "—")
        print(f"  {recipe:36s} {(ev if has else 'none'):>12}  {claimed}")
    except RuntimeError as exc:
        print(f"  {recipe:36s} {'rejected':>12}  {str(exc)[:34]}")

rule()
wrap("""Read the evidence column, not just yes/no. Where the encoder can certify a true
property it will, so several recipes report CheckedDescriptor. What distinguishes
unsigned delta is Structural: no trust required. And the framed row is the paper's
recurring case — the fact survives the frame, the cheap probe does not.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.4 Figure — a compiled plan, drawn. Nodes are drawn from the REAL plan;
# a node is blue when it cites a fact (authorised fast path) and grey when it
# is unconditional. The label under each node is its byte closure.
# ═════════════════════════════════════════════════════════════════════════════
def draw_plan(values, recipe, query, title):
    plan = witness("compile", values=values, recipe=recipe, query=query)
    nodes = plan["nodes"]

    fig, ax = plt.subplots(figsize=(10.2, 2.9))
    ax.set_axis_off(); ax.grid(False)
    n = len(nodes)
    xs = np.linspace(0.5 / n, 1 - 0.5 / n, n)

    for x, node in zip(xs, nodes):
        cited = node["cites"] != "unconditional"
        edge = BLUE if cited else GRID
        face = "#EAF2FC" if cited else "white"
        op = node["op"].split(" {")[0]
        ax.add_patch(mpl.patches.FancyBboxPatch(
            (x - 0.085, 0.46), 0.17, 0.30,
            boxstyle="round,pad=0.012,rounding_size=0.03",
            linewidth=1.7, edgecolor=edge, facecolor=face))
        ax.text(x, 0.61, op, ha="center", va="center", fontsize=8.2, color=INK)
        closure = (f"{node['closure_bytes']} B"
                   if node["closure"] == "exact" else "runtime-refined")
        ax.text(x, 0.38, closure, ha="center", fontsize=7.2, color=MUTED)
        if cited:
            ax.text(x, 0.28, node["cites"].split("::")[-1][:26], ha="center",
                    fontsize=7.0, color=BLUE, style="italic")
        ax.text(x, 0.80, node["guarantee"].split("(")[0], ha="center",
                fontsize=7.0, color=MUTED)

    for x1, x2 in zip(xs, xs[1:]):
        ax.annotate("", xy=(x2 - 0.088, 0.61), xytext=(x1 + 0.088, 0.61),
                    arrowprops=dict(arrowstyle="-|>", color=MUTED, linewidth=1.2))

    ax.set_xlim(0, 1); ax.set_ylim(0.20, 0.92)
    ax.set_title(title, loc="left", color=INK)
    plt.tight_layout(); plt.show()
    return plan


rule("§3.4 — the same query on two columns, plans drawn from the real compiler")
BIGQ2 = {"kind": "filter_between", "low": 1000, "high": 1100}
RUN2 = list(range(0, 8192, 2))

p1 = draw_plan(RUN2, "UnsignedDelta(256, BitPack)", BIGQ2,
               "ordered column — SearchMonotone is authorised (blue node cites the fact)")
p2 = draw_plan(sorted(RUN2, key=lambda v: (v * 7919) % 4096), "Delta(256, BitPack)", BIGQ2,
               "shuffled column — no order fact, every node unconditional")

print(f"  {'plan':34s} {'nodes':>6s} {'cited':>6s} {'output':>16s}")
for label, p in [("ordered / UnsignedDelta", p1), ("shuffled / Delta", p2)]:
    cited = sum(1 for n in p["nodes"] if n["cites"] != "unconditional")
    print(f"  {label:34s} {p['node_count']:>6} {cited:>6} {p['output']:>16}")
wrap("""Blue means the node named a fact and the checker accepted the citation. Grey
means unconditional — the step is always legal because it assumes nothing. A plan made
entirely of grey nodes is a scan, and that is the compiler working correctly, not
failing.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3.3 Figure — the evidence matrix. Which recipe yields which evidence class
# for the order property, on an ordered vs an unordered column.
# Form: categorical status grid. Colour carries a STATE (structural / checked /
# absent), so it ships with text in every cell — never colour alone.
# ═════════════════════════════════════════════════════════════════════════════
GRID_RECIPES = ["BitPack", "For(BitPack)", "Delta(256, BitPack)",
                "UnsignedDelta(256, BitPack)", "Rle(64, BitPack)",
                "Dictionary(BitPack)", "Frame(UnsignedDelta(256, BitPack))"]
COLUMNS = [("ordered", X), ("unordered", UNORDERED)]

STATE = {"Structural": (BLUE, "S"), "CheckedDescriptor": (ORANGE, "C"), "none": ("#FFFFFF", "—")}
cellstate = []
for recipe in GRID_RECIPES:
    row = []
    for _, values in COLUMNS:
        try:
            got = witness("facts", values=values, recipe=recipe)
            ev = next((f["evidence"].split("(")[0] for f in got["facts"]
                       if "NonDecreasing" in f["property"]), "none")
        except RuntimeError:
            ev = "none"
        row.append(ev if ev in STATE else "none")
    cellstate.append(row)

fig, ax = plt.subplots(figsize=(6.6, 0.46 * len(GRID_RECIPES) + 1.4))
ax.set_axis_off(); ax.grid(False)
for r, recipe in enumerate(GRID_RECIPES):
    y = len(GRID_RECIPES) - r - 1
    ax.text(-0.06, y + 0.5, recipe, ha="right", va="center", fontsize=8.2, color=INK)
    for c, (label, _) in enumerate(COLUMNS):
        colour, glyph = STATE[cellstate[r][c]]
        ax.add_patch(mpl.patches.Rectangle((c, y), 0.94, 0.9, facecolor=colour,
                                           edgecolor=GRID, linewidth=1.0))
        ink = "white" if cellstate[r][c] != "none" else MUTED
        ax.text(c + 0.47, y + 0.45, glyph, ha="center", va="center",
                fontsize=10, color=ink, weight="bold")
for c, (label, _) in enumerate(COLUMNS):
    ax.text(c + 0.47, len(GRID_RECIPES) + 0.12, label, ha="center", fontsize=8.6, color=INK)
ax.set_xlim(-2.4, len(COLUMNS)); ax.set_ylim(0, len(GRID_RECIPES) + 0.5)
ax.set_title("§3.3 — evidence for NonDecreasing:  S structural · C checked · — none",
             loc="left", color=INK)
plt.tight_layout(); plt.show()

print(f"  {'recipe':36s} {'ordered':>18s} {'unordered':>18s}")
print(f"  {'-'*36} {'-'*18} {'-'*18}")
for recipe, row in zip(GRID_RECIPES, cellstate):
    print(f"  {recipe:36s} {row[0]:>18s} {row[1]:>18s}")
wrap("""Read the ordered column downward: almost every recipe yields SOME evidence,
because the encoder certifies what it can verify. Only unsigned delta yields
Structural — free and unfalsifiable. Now read the unordered column: nearly everything
empties out, because there is no true property to certify. That contrast is the
mechanism, and it is why the paper distinguishes evidence classes rather than just
listing properties.""", indent="  ")

---
## §4 Evaluation

Everything above is mechanism. From here on the numbers are **measured**, loaded from
the artifact's committed CSVs — the same files the paper's macros are generated from.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §4.1 — Workload and reference points.
# ═════════════════════════════════════════════════════════════════════════════
rule("§4.1 — the study's shape")

print(f"""  corpus      ClickBench, TPC-H lineitem, Public BI, NAB, UCI household power, NYC taxi
  scale       {claim('WitPairs')} selector/value pairs, {claim('WitColumns')} columns,
              {claim('WitSources')} sources, {claim('WitRowsPerColumn')} rows per column
  cells       {claim('WitCells')} distinct query cells

  query shape BETWEEN window at a target selectivity from 0.1% to 50%,
              then SUM over the matching rows
  protocol    every arm must return IDENTICAL rows and sums before it is timed
  statistics  p25/median/p75 over calibrated repetitions;
              bootstrap intervals resample SOURCES, not correlated cells
""")

wrap("""The two-stage decomposition is what makes the result interpretable: DISCOVERY
finds matching row positions, AGGREGATION consumes the value column at them. Reporting
only end-to-end time would conflate a fact that removes search work with a decoder
that happens to be fast.""", indent="  ")

print("\n  Reference points (external calibration, not a universal opponent):")
for name, detail in [
    ("parquet_full",            "stock Arrow/Parquet full decode"),
    ("parquet_row_filter",      "stock Arrow/Parquet row-filter path"),
    ("parquet_boundary_search", "our reader for Parquet's ColumnIndex boundary-order flag"),
]:
    print(f"    {name:24s} {detail}")
wrap("""The boundary-order reader is deliberately the STRONGER reference: it loads the
ColumnIndex, searches page minima and maxima, and filters only boundary pages. Beating
a reader that ignores the flag would prove little.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §4.1 (b) — The two encoding policies. This is the only thing that differs
# between the two Witness arms; values, compiler and queries are identical.
# ═════════════════════════════════════════════════════════════════════════════
rule("§4.1 — size-selected vs access-ready")

print(f"""  size-selected   encode every applicable candidate, keep the SMALLEST page.
                  Typically ends up inside a Zstd frame.
                  -> minimal bytes, opaque to selective access.

  access-ready    keep the smallest candidate with BOUNDED LOCAL dependencies.
                  Opaque frames excluded; delta restart intervals shortened.
                  -> more bytes, but a probe reaches a row without bulk decode.

  measured price  {claim('WitAccessPremium')}x the bytes  (canonical aggregate)
  measured gain   {claim('WitDirectOverStorage')}x the time (cell median)
                  {claim('WitSourceDirectStorageMedian')}x over source medians
""")
wrap("""This is a causal comparison in the strict sense: one policy differs, everything
else is held fixed. Whatever the difference in latency is, it is attributable to the
layout choice and not to a different codec, a different query, or a different
compiler.""", indent="  ")

---
## §5 Results

Three findings before the detail:

1. usable order evidence occurs in a meaningful minority of columns — and **the free
   proof and the usable proof are not the same object**;
2. an authorized search removes most of the predicate-discovery work at low
   selectivity, and that share shrinks as selections widen;
3. keeping access costs roughly double the bytes, so **the fastest layout is not the
   smallest one**.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.1 Q1 — How often is evidence available? The census, and its zero row.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.1 Q1 — fact incidence over real columns")

census_rows = load_csv("invariant_census/summary.csv",
                       fallback_rows=[{"metric": "columns", "value": "109"}])
census = {r["metric"]: r["value"] for r in census_rows}

table = [
    ("Global monotone (true)",        claim('WitCensusMonotoneColumns'), "—"),
    ("Structural global proof",       claim('WitCensusMonotoneColumns'), claim('WitCensusStructuralPremiumMedian')),
    ("Structural global + access",    claim('WitCensusStructuralAccessReady'), "0 (structural)"),
    ("Checked global + access",       claim('WitCensusCheckedAccessReady'), f"{claim('WitCertificateHeaderBytes')} B desc."),
    ("Structural piecewise + access", claim('WitCensusPiecewiseAccessReady'), "—"),
    ("Monotone pages (1024 rows)",     claim('WitCensusMonotonePages'), "—"),
    ("Rows in monotone runs >= 128",   claim('WitCensusSegmentFraction'), "—"),
    ("Sorted dict mapping possible",  claim('WitCensusOrderMapping'), claim('WitCensusOrderMappingPremium')),
    ("Dictionary-friendly domain",    claim('WitCensusSmallDomain'), "—"),
]
print(f"  {'Fact / usability':34s} {'Share %':>10s} {'Premium':>16s}")
print(f"  {'-'*34} {'-'*10} {'-'*16}")
for label, pct, premium in table:
    print(f"  {label:34s} {pct:>10s} {premium:>16s}")

rule()
wrap(f"""The informative row is the zero. A STRUCTURAL global proof needs one restart
block spanning the whole column; access-ready needs a restart every 256 rows. Beyond
256 rows the two cannot both hold — so {claim('WitCensusStructuralAccessReady')}% is a
consequence of the definitions, not a survey result. The finding is what survives:
weaker facts stay usable where the strongest cannot. Piecewise evidence is seekable,
and a checked descriptor recovers seekable global order at the cost of a trust
boundary instead of a zero-byte proof.""", indent="  ")
wrap("""The free proof and the usable proof differ, and a query needs the second.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.1 Q1 (visual) — incidence is far from uniform across corpus families.
# Form: horizontal bars. Job = magnitude comparison across named groups,
# so one hue, sorted by value, direct-labelled. No second series -> no legend.
# ═════════════════════════════════════════════════════════════════════════════
sources = load_csv("invariant_census/sources.csv", fallback_rows=[
    {"group": "nab", "columns": "48", "global_monotone_columns": "24"},
    {"group": "publicbi", "columns": "17", "global_monotone_columns": "1"},
])

groups = {}
for row in sources:
    g = groups.setdefault(row["group"], {"columns": 0, "monotone": 0})
    g["columns"] += int(row["columns"])
    g["monotone"] += int(row["global_monotone_columns"])

names = sorted(groups, key=lambda g: groups[g]["monotone"] / max(groups[g]["columns"], 1))
pcts = [100 * groups[g]["monotone"] / max(groups[g]["columns"], 1) for g in names]
counts = [groups[g]["columns"] for g in names]

fig, ax = plt.subplots(figsize=(7.4, 0.42 * len(names) + 1.5))
bars = ax.barh(names, pcts, color=BLUE, height=0.62)
for bar, pct, n in zip(bars, pcts, counts):
    ax.text(bar.get_width() + 1.2, bar.get_y() + bar.get_height()/2,
            f"{pct:.0f}%  (n={n})", va="center", fontsize=8, color=MUTED)
ax.set_xlabel("columns with global monotonicity (%)")
ax.set_xlim(0, max(pcts) * 1.35 + 6)
ax.set_title("§5.1 — fact incidence is highly uneven across corpus families", loc="left")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

print(f"  {'group':14s} {'columns':>8s} {'monotone':>9s} {'share':>7s}")
for g in reversed(names):
    d = groups[g]
    print(f"  {g:14s} {d['columns']:>8d} {d['monotone']:>9d} "
          f"{100*d['monotone']/max(d['columns'],1):>6.1f}%")
wrap("""NAB sensor columns are monotone about half the time; Public BI and Yellow Taxi
almost never. A single corpus-wide average would hide exactly the variation that
decides whether this mechanism is worth deploying on YOUR data.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.2 Control — familiar metadata and membership. Witness must not look good
# merely because ordinary metadata was withheld.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.2 — membership certificates under the same byte budget")

cert = load_csv("certificate_study/summary.csv", fallback_rows=[])
wanted = [("bloom", "eq_absent"), ("bloom", "eq_rare"),
          ("bloom", "eq_frequent"), ("bloom", "in_mixed"),
          ("sparse_fence", "eq_rare")]

print(f"  {'plan':13s} {'query':12s} {'src':>4s} {'cand.':>7s} {'bytes/scan':>11s} {'latency':>8s}")
print(f"  {'-'*13} {'-'*12} {'-'*4} {'-'*7} {'-'*11} {'-'*8}")
for plan, query in wanted:
    row = next((r for r in cert if r["plan"] == plan and r["query"] == query), None)
    if row is None:
        print(f"  {plan:13s} {query:12s}  (dummy — CSV absent)")
        continue
    print(f"  {plan:13s} {query:12s} {row['sources']:>4s} "
          f"{float(row['candidate_fraction_median']):>7.3f} "
          f"{float(row['modeled_bytes_over_scan_median']):>11.3f} "
          f"{float(row['source_latency_ratio_median']):>7.3f}x")

rule()
wrap("""The expected boundary, and the paper reports it as such. Bloom filters reject
ABSENT values outright and sharply reduce candidates for RARE ones. For FREQUENT and
mixed-IN probes almost every block becomes a candidate and the metadata overhead
exceeds a plain scan — the modeled-bytes column crosses 1.0. Evidence is
query-specific: a certificate should be used only for the question its guarantee can
answer. This section exists to stop the reader crediting Witness for beating a
strawman.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.2 Figure — the control, drawn. A certificate is only worth its bytes when
# it removes candidates. Both axes are ratios to a plain scan; the parity lines
# are where the certificate stops paying for itself.
# Two series (candidate fraction, modeled bytes) -> legend + printed table.
# ═════════════════════════════════════════════════════════════════════════════
rows = [(r["plan"], r["query"], float(r["candidate_fraction_median"]),
         float(r["modeled_bytes_over_scan_median"]),
         float(r["source_latency_ratio_median"]))
        for r in cert
        if (r["plan"], r["query"]) in
           {("bloom","eq_absent"),("bloom","eq_rare"),("bloom","eq_frequent"),
            ("bloom","in_mixed"),("sparse_fence","eq_rare")}]

if rows:
    labels = [f"{p.replace('_',' ')}\n{q.replace('_',' ')}" for p, q, *_ in rows]
    cand = [r[2] for r in rows]
    byts = [r[3] for r in rows]
    idx = np.arange(len(rows))

    fig, ax = plt.subplots(figsize=(8.4, 3.8))
    ax.bar(idx - 0.19, cand, width=0.36, color=BLUE, label="candidate fraction")
    ax.bar(idx + 0.19, byts, width=0.36, color=ORANGE, label="modeled bytes / scan")
    ax.axhline(1.0, color=MUTED, ls=":", lw=1.3)
    ax.text(-0.45, 1.04, "parity with a plain scan", fontsize=7.4, color=MUTED)
    ax.set_xticks(idx); ax.set_xticklabels(labels, fontsize=7.6)
    ax.set_ylabel("ratio to scan")
    ax.set_title("§5.2 — a certificate earns its bytes only when it removes candidates",
                 loc="left")
    ax.legend(); ax.grid(axis="x", visible=False)
    plt.tight_layout(); plt.show()

    print(f"  {'plan':14s} {'query':12s} {'candidates':>11s} {'bytes/scan':>11s} {'latency':>8s}")
    print(f"  {'-'*14} {'-'*12} {'-'*11} {'-'*11} {'-'*8}")
    for p, q, c, b, l in rows:
        flag = "  <- costs more than scanning" if b > 1.0 else ""
        print(f"  {p:14s} {q:12s} {c:>11.3f} {b:>11.3f} {l:>7.3f}x{flag}")

wrap("""The two bars move together, and that is the whole story. Where the candidate
fraction collapses (absent, rare) the certificate is cheap and effective. Where it
approaches 1.0 (frequent, mixed IN) every block survives, the metadata is pure
overhead, and the orange bar crosses parity — the certificate now costs more than the
scan it was meant to avoid.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.3 Q2 — What work does an order fact remove? Discovery vs aggregation.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.3 Q2 — the decomposition")

print(f"""  discovery share, median over all cells      {claim('WitDiscoveryShare')}
    at 0.1% target selectivity                 {claim('WitDiscoveryShareTenth')}
    at 1%                                      {claim('WitDiscoveryShareOnePct')}
    at 10%                                     {claim('WitDiscoveryShareTenPct')}
    at 50%                                     {claim('WitDiscoveryShareFiftyPct')}

  generated search beats Witness's OWN scan in {claim('WitFilterBeatsScan')}/{claim('WitCells')} cells
  complete access-ready pipeline vs its scan   {claim('WitCompleteOverScan')}x
""")

wrap("""A boundary search touches about the same few pages regardless of match count,
so its SHARE falls as aggregation grows with the number of matched rows. That is the
familiar crossover that favours an index scan over a sequential one, measured rather
than assumed.""", indent="  ")

print(f"""
  And the honest half of the decomposition:
    with matching positions supplied for free, Witness's aggregation on the
    size-selected layout is SLOWER than mature Parquet row-selection:
        size-selected  {claim('WitKnownOverParquet')}x
        access-ready   {claim('WitKnownDirectOverParquet')}x
""")
wrap("""So the contribution is in DISCOVERY, not in decoding. The paper does not claim
to explain the remaining aggregation deficit: tuning the bit-unpack kernel left it
essentially unchanged, and removing a per-value heap allocation cut it by about a
third without closing it. It is therefore neither decode throughput nor allocation
alone.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.3 Figure 2 (fig:effect) — the effect, conditioned on the certificate.
# Form: (a) class-median ratio vs selectivity, (b) ECDF of per-cell ratios.
# Two series = identity -> categorical hues, legend present, dashed parity line.
# Log x on both panels because selectivity and ratios span decades.
# ═════════════════════════════════════════════════════════════════════════════
def parse_plot(macro):
    """The manifest stores plot series as (x,y)(x,y)... — parse back to arrays."""
    raw = claim(macro)
    if raw == "<dummy>":
        return np.array([0.1, 1, 10, 50]), np.array([0.15, 0.2, 0.5, 1.0])
    pts = [p for p in raw.replace(")(", ")|(").split("|") if p]
    xs, ys = [], []
    for p in pts:
        a, b = p.strip("()").split(",")
        xs.append(float(a)); ys.append(float(b))
    return np.array(xs), np.array(ys)

cx_m, cy_m = parse_plot("WitCurveMonotonePlot")
cx_n, cy_n = parse_plot("WitCurveNoFactPlot")
ex_m, ey_m = parse_plot("WitEcdfMonotonePlot")
ex_n, ey_n = parse_plot("WitEcdfNoFactPlot")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.5))

ax1.plot(cx_m, cy_m, marker="o", ms=5, lw=2, color=BLUE, label="monotone fact holds")
ax1.plot(cx_n, cy_n, marker="^", ms=6, lw=2, ls="--", color=ORANGE, label="no fact (scan)")
ax1.axhline(1.0, color=MUTED, ls=":", lw=1.2)
ax1.text(cx_m[0]*1.05, 1.06, "parity", fontsize=7.5, color=MUTED)
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("target selectivity (%)"); ax1.set_ylabel("access-ready / Parquet")
ax1.set_title("(a) class-median latency ratio", loc="left")
ax1.legend(loc="upper left", fontsize=8)

ax2.step(ex_m, ey_m, where="post", lw=2, color=BLUE, label="monotone fact holds")
ax2.step(ex_n, ey_n, where="post", lw=2, ls="--", color=ORANGE, label="no fact (scan)")
ax2.axvline(1.0, color=MUTED, ls=":", lw=1.2)
ax2.set_xscale("log"); ax2.set_ylim(0, 1.02)
ax2.set_xlabel("per-cell ratio: access-ready / Parquet"); ax2.set_ylabel("fraction of cells")
ax2.set_title("(b) distribution over all cells", loc="left")
ax2.legend(loc="upper left", fontsize=8)

plt.tight_layout(); plt.show()

print(f"  below the parity line / left of it = the compiled plan is faster")
print(f"  monotone class cells : {len(ex_m):3d}    fraction below parity: {(ex_m < 1).mean():.0%}")
print(f"  no-fact class cells  : {len(ex_n):3d}    fraction below parity: {(ex_n < 1).mean():.0%}")
wrap("""The monotone class gains most at small selections and converges toward parity by
construction — as the selection widens there is simply less search to remove. The
no-fact class straddles parity everywhere, which is what 'the fact is doing the work'
looks like when it is true.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.3 (b) — The certificate class split: the clearest causal reading.
# Assignment is OUTCOME-BLIND — the encoded certificate decides the class,
# not the measured latency.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.3 — split by what the encoding certifies, before any timing")

split = load_csv("predicate_pipeline/certificate_summary.csv", fallback_rows=[])
print(f"  {'class':16s} {'src':>4s} {'cells':>6s} {'vs boundary':>12s} {'vs equal-page':>14s} {'wins':>7s}")
print(f"  {'-'*16} {'-'*4} {'-'*6} {'-'*12} {'-'*14} {'-'*7}")
for row in split:
    label = "monotone" if row["non_decreasing"] == "true" else "no fact"
    print(f"  {label:16s} {row['sources']:>4s} {row['predicate_cells']:>6s} "
          f"{float(row['source_median_direct_gen_over_parquet_boundary']):>11.2f}x "
          f"{float(row['source_median_direct_gen_over_equal_page_parquet_boundary']):>13.2f}x "
          f"{row['direct_sources_beating_parquet_boundary']:>3s}/{row['sources']:<3s}")

rule()
wrap("""The class is assigned by the certificate the encoder wrote, not by how fast the
query turned out to be — so this is not a post-hoc split. The gap between the two rows
is the mechanism's signature. The paper is careful to add that the no-fact class is
small and directional rather than a controlled population.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.4 Q3 — What does selective access cost? The storage/access frontier.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.4 Q3 — the price of keeping access")

print(f"""  access-ready vs size-selected
    latency  {claim('WitDirectOverStorage')}x  (cell median)
             {claim('WitSourceDirectStorageMedian')}x  over source medians,
             CI [{claim('WitSourceDirectStorageCiLow')}, {claim('WitSourceDirectStorageCiHigh')}]
    bytes    {claim('WitAccessPremium')}x  (canonical aggregate)

  across the {claim('WitPremiumGroups')} corpus groups the byte premium ranges
    {claim('WitPremiumGroupLow')}x .. {claim('WitPremiumGroupHigh')}x
  -> it is NOT a constant header charge; it depends on what the column is.

  external context vs the sortedness-aware Parquet reference
    size-selected policy wins {claim('WitStorageBeatsBoundary')}/{claim('WitCells')} cells (median {claim('WitStorageOverBoundary')}x)
    access-ready policy wins  {claim('WitDirectBeatsBoundary')}/{claim('WitCells')} cells (median {claim('WitDirectOverBoundary')}x)
""")
wrap("""A byte-minimising encoder can choose the wrong point for selective work. The
contribution is to expose that choice, not to erase the bytes — the smaller delivered
closure is purchased, not free.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.4 Figure 3 (fig:frontier) — one mark per source: bytes paid vs time saved.
# Form: scatter. Both axes are ratios spanning decades -> log-log.
# Single series -> no legend needed; the title names it. Quadrants annotated.
# ═════════════════════════════════════════════════════════════════════════════
fx, fy = parse_plot("WitFrontierPlot")

fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.scatter(fx, fy, s=42, facecolors="none", edgecolors=BLUE, linewidths=1.6, zorder=3)
ax.axhline(1.0, color=MUTED, ls=":", lw=1.2, zorder=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("byte premium  (access-ready / size-selected)")
ax.set_ylabel("latency ratio  (access-ready / size-selected)")
ax.set_title("§5.4 — the storage/access frontier, one mark per source", loc="left")
ax.text(ax.get_xlim()[0]*1.05, 1.08, "parity — no time saved", fontsize=7.5, color=MUTED)
ax.text(ax.get_xlim()[0]*1.05, min(fy)*1.1, "faster, but paid for in bytes",
        fontsize=7.5, color=MUTED)
plt.tight_layout(); plt.show()

print(f"  sources plotted            : {len(fx)}")
print(f"  faster than size-selected  : {(fy < 1).sum()}/{len(fy)}")
print(f"  byte premium  min/med/max  : {fx.min():.2f}x / {np.median(fx):.2f}x / {fx.max():.2f}x")
print(f"  latency ratio min/med/max  : {fy.min():.3f}x / {np.median(fy):.3f}x / {fy.max():.3f}x")
wrap("""Most points trade a low single-digit byte multiple for materially lower
selective-query latency — but neither axis dominates, and the marks near the top show
sources where the extra bytes bought nothing. That spread IS the result; a single
average would erase it.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.5 — Cold I/O and scale. Fewer logical bytes is not fewer physical reads.
# ═════════════════════════════════════════════════════════════════════════════
rule("§5.5 — cold XFS, random order: the read-schedule boundary")

storage = load_csv("real_access/storage_scan.csv", fallback_rows=[])
cold = [r for r in storage if r["storage"] == "workspace_mount" and r["filesystem"] == "xfs"
        and r["order"] == "random" and r["cache_state"] == "cold"]

if cold:
    full = next(r for r in cold if r["policy"] == "full_file")
    base = float(full["median_ns"])
    print(f"  {'policy':16s} {'read calls':>11s} {'scheduled MiB':>14s} {'vs full read':>13s}")
    print(f"  {'-'*16} {'-'*11} {'-'*14} {'-'*13}")
    for r in sorted(cold, key=lambda r: -float(r["median_ns"])):
        print(f"  {r['policy']:16s} {r['read_calls']:>11s} "
              f"{float(r['scheduled_bytes'])/1048576:>14.2f} "
              f"{float(r['median_ns'])/base:>12.2f}x")
else:
    print("  (dummy — storage_scan.csv absent)")

print(f"""
  required closure {claim('WitColdRequiredMb')} MiB out of a {claim('WitColdFileMb')} MiB file
  per-page   {claim('WitColdPageCalls')} calls -> {claim('WitColdPageOverFull')}x a full-file read
  coalesced  {claim('WitColdCoalescedCalls')} calls -> {claim('WitColdCoalescedOverFull')}x
""")
wrap("""The plan that reads the FEWEST bytes is the slowest, because it issues the most
requests. Closure leaves readahead and call count to a storage-aware scheduler; it
does not by itself produce less physical I/O. This is the paper's most important
negative result and it is why 'delivered bytes' and 'time' are reported as separate
counters rather than one proxy for the other.""", indent="  ")

print(f"\n  Scale check — the same frozen rules at 16Ki rows instead of 128Ki:")
print(f"    access-ready vs boundary : {claim('WitSixteenKDirectOverBoundary')}x  "
      f"(vs {claim('WitDirectOverBoundary')}x at full scale)")
print(f"    discovery share          : {claim('WitSixteenKDiscoveryShare')}   "
      f"(vs {claim('WitDiscoveryShare')})")
wrap("""Headline numbers move only slightly with an 8x change in row count, which
supports an algorithmic explanation rather than a cache-residency artifact.""",
     indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5.5 Figure — the cold-I/O inversion, drawn. Fewer bytes scheduled does not
# mean less time; the number of READ CALLS is what dominates.
# Form: scatter, calls (log x) against time relative to a full read.
# Single series -> no legend; points are direct-labelled with their policy.
# ═════════════════════════════════════════════════════════════════════════════
if cold:
    calls = [int(r["read_calls"]) for r in cold]
    ratio = [float(r["median_ns"]) / base for r in cold]
    sched = [float(r["scheduled_bytes"]) / 1048576 for r in cold]
    names = [r["policy"] for r in cold]

    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    ax.scatter(calls, ratio, s=[22 + 5 * m for m in sched], facecolors="none",
               edgecolors=BLUE, linewidths=1.7, zorder=3)
    for c, y, n, m in zip(calls, ratio, names, sched):
        ax.annotate(f"{n}\n{m:.1f} MiB", (c, y), textcoords="offset points",
                    xytext=(9, -3), fontsize=7.4, color=MUTED)
    ax.axhline(1.0, color=MUTED, ls=":", lw=1.3)
    ax.set_xscale("log")
    ax.set_xlabel("read calls issued (log)")
    ax.set_ylabel("median time / full-file read")
    ax.set_title("§5.5 — the plan that reads fewest bytes issues the most calls, and loses",
                 loc="left")
    plt.tight_layout(); plt.show()

    print(f"  {'policy':16s} {'calls':>7s} {'scheduled MiB':>14s} {'vs full read':>13s}")
    print(f"  {'-'*16} {'-'*7} {'-'*14} {'-'*13}")
    for r in sorted(cold, key=lambda r: -float(r["median_ns"])):
        print(f"  {r['policy']:16s} {r['read_calls']:>7s} "
              f"{float(r['scheduled_bytes'])/1048576:>14.2f} "
              f"{float(r['median_ns'])/base:>12.2f}x")

wrap("""Marker area is bytes scheduled. The per-page policy sits top-left: it schedules
the least data and takes the longest, because it pays for over a thousand separate
requests. Coalescing 4 KiB gaps moves it to the bottom-right — more bytes, fewer calls,
less time. Any evaluation that reported only logical bytes would have ranked these
policies exactly backwards.""", indent="  ")

---
## §6 Generality Beyond Monotonicity

Everything so far turns on one fact family. If the contract is a calculus rather than
a special case for order, other facts must license other plans.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §6 ¶1-2 — Dictionary range translation. Order-preserving by CONSTRUCTION,
# so the rewrite needs no trust and no row reads.
# ═════════════════════════════════════════════════════════════════════════════
rule("§6 — plan 1: TranslateDictionaryRange")

print(f"""  scope    {claim('WitDictPairs')} of {claim('WitPairs')} pairs have a selector the recipe search
           chose — unprompted — to dictionary-encode; {claim('WitDictCells')} cells
  bound    largest dictionary {claim('WitDictEntries')} entries -> at most {claim('WitDictProbes')} probes
           against {claim('WitRowsPerColumn')} rows
  measured median {claim('WitDictLatency')}x the scan reference (END-TO-END)
""")
wrap("""The honest caveat the paper insists on: translation bounds the PREDICATE work to
two binary searches, but the code stream is still scanned unless it proves its own
order. So the reported ratio is the end-to-end figure, not the translation stage in
isolation. Quoting the probe-count reduction as if it were the speedup would be the
easy overclaim here, and the paper explicitly declines it.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §6 ¶3 — Run-length counting. A different fact family, a different plan:
# COUNT(y=v) in O(runs) rather than O(rows).
# ═════════════════════════════════════════════════════════════════════════════
rule("§6 — plan 2: CountRuns")

# DUMMY DATA: a small run-structured column, to show the mechanism exactly.
runs = [(7, 4), (3, 5), (7, 2), (9, 6)]          # (value, run length)
expanded = [v for v, n in runs for _ in range(n)]
target = 7

by_rows = sum(1 for v in expanded if v == target)
by_runs = sum(n for v, n in runs if v == target)
print(f"  runs      : {runs}")
print(f"  expanded  : {expanded}   ({len(expanded)} rows, {len(runs)} runs)")
print(f"  COUNT(y={target}) over rows : {by_rows}   ({len(expanded)} row visits)")
print(f"  COUNT(y={target}) over runs : {by_runs}   ({len(runs)} run visits)")
assert by_rows == by_runs, "CountRuns must agree with the brute-force count"
print(f"  identical: {by_rows == by_runs}    reduction: {len(expanded)}/{len(runs)} = "
      f"{len(expanded)/len(runs):.1f}x fewer visits")

print(f"""
  On the real corpus:
    {claim('WitRleColumns')} of {claim('WitCensusColumns')} census columns clear a uniform 4x run/row threshold
    median reduction {claim('WitRleReductionMedian')}x
    CountRuns reaches median {claim('WitRleLatency')}x the CountExact fallback
""")
wrap("""Selection is a THRESHOLD on measured reduction, not a hand-picked list — every
exact-integer census column with at least a 4x reduction enters, so the reported
median is not a best-case cherry pick. Any other encoding falls back to an explicit
scan.""", indent="  ")

print("\n  A measurement subtlety the paper calls out:")
wrap("""they do NOT time against a per-row walk of the run-length column itself. That
path re-enters the sparse run index once per row, which would credit the plan with an
access-pattern artifact rather than the fact it is meant to demonstrate.""",
     indent="      ")

---
## §7 Related Work

Where this sits among systems that compose codecs or persist shape facts.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §7 — positioning. The axis that separates Witness is narrow and stated plainly.
# ═════════════════════════════════════════════════════════════════════════════
rule("§7 — neighbours, and the one axis of difference")

neighbours = [
    ("FastLanes",      "cascaded encodings, partial decompression",
     "supplies the substrate; operators are hand-written per codec"),
    ("Vortex",         "compute over compressed vectors",
     "same — no shared soundness contract or generic fallback"),
    ("BtrBlocks",      "cascade selection by sampling",
     "same"),
    ("CodecDB",        "picks an encoding per column, executes on the encoded form",
     "CLOSEST premise; but operators chosen by a cost model, not licensed by a "
     "stated property — nothing forces a fallback when the property fails"),
    ("Data Blocks",    "positional SMA per compressed block narrows scan ranges",
     "closest existing case of a persisted fact steering access; but ONE fixed "
     "structure, not a typed derivation over composed codecs"),
    ("White-box compr.", "learned table expressions and execution",
     "closest conceptual precursor to typed facts"),
    ("SMA / zone maps / ORC index / Parquet boundary-order",
     "each persists a particular fact",
     "none types a fact with its evidence, or requires a plan to cite one"),
]
for name, idea, relation in neighbours:
    print(f"\n  {name}")
    wrap(f"idea: {idea}", indent="      ")
    wrap(f"vs Witness: {relation}", indent="      ")

rule()
wrap("""The paper then narrows its own claim further than a reader might expect: this
comparison covers only storage formats, so the novelty is narrower than the framing
suggests. Witness is a storage-specific instance of deriving conservative facts from
structure (abstract interpretation) and licensing a plan from physical properties
without re-deriving them (interesting orders). And its trust model is WEAKER than
proof-carrying code or proof-carrying data: both treat the producer as untrusted and
have the consumer check an attached proof, whereas a checksum authenticates BYTES,
not the truth of the fact.""", indent="  ")

---
## §8 Limitations and Outlook

The section that decides whether the rest is trustworthy. It also contains the
statistical machinery, which is worth running rather than reading.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §8 (a) — THE ESTIMATOR. Why "median" must mean the standard median.
# This is not pedantry: every source in the study has an EVEN cell count, so a
# single-central-element shortcut biases every per-source estimate.
# ═════════════════════════════════════════════════════════════════════════════
rule("§8 — the median convention, and why it is load-bearing")

def median_standard(v):
    s = sorted(v); n = len(s); m = n // 2
    return s[m] if n % 2 else (s[m-1] + s[m]) / 2

def median_upper(v):                 # the shortcut: values[len//2]
    return sorted(v)[len(v)//2]

sample = [0.20, 0.40, 0.60, 1.00]
print(f"  sample                {sample}")
print(f"  standard median       {median_standard(sample):.2f}   (mean of the two central values)")
print(f"  upper-central element {median_upper(sample):.2f}   <- biased HIGH")

# On lower-is-better ratios, the bias runs against the system under test.
src = load_csv("predicate_pipeline/source_summary.csv", fallback_rows=[])
if src:
    per_source = [float(r["complete_direct_gen_over_parquet_boundary_median"]) for r in src]
    cells = [int(r["predicate_cells"]) for r in src]
    even = sum(1 for c in cells if c % 2 == 0)
    print(f"\n  sources                        {len(cells)}")
    print(f"  sources with an EVEN cell count {even}/{len(cells)}  <- the shortcut always bites")
    print(f"  headline, standard median       {median_standard(per_source):.3f}")
    print(f"  headline, upper-central         {median_upper(per_source):.3f}")

rule()
wrap("""Because the ratio is lower-is-better, the shortcut made the system look WORSE
than it is — a conservative error, but an error. It is corrected in the artifact, and
regression tests now fail if anyone reintroduces it. The lesson generalises: an
estimator that looks reasonable can be systematically wrong when every group happens
to share a parity.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §8 (b) — THE POPULATION CLAIM. The corpus is nested: NAB contributes several
# columns per archive family, so 32 "sources" are not 32 independent draws.
# ═════════════════════════════════════════════════════════════════════════════
rule("§8 — three ways to count, and why the answer moves")

if src:
    families = {}
    for r in src:
        fam = r["source"].split("/")[0]
        families.setdefault(fam, []).append(
            float(r["complete_direct_gen_over_parquet_boundary_median"]))

    print(f"  {'family':26s} {'n':>3s} {'median':>8s}")
    print(f"  {'-'*26} {'-'*3} {'-'*8}")
    for fam in sorted(families, key=lambda f: -len(families[f])):
        print(f"  {fam[:26]:26s} {len(families[fam]):>3d} {median_standard(families[fam]):>8.3f}")

    M = (1 << 64) - 1
    def xorshift(state):
        state ^= (state << 13) & M; state &= M
        state ^= state >> 7
        state ^= (state << 17) & M; state &= M
        return state

    def boot_flat(values, reps=2000):
        st, est = 0x6a09e667f3bcc909, []
        for _ in range(reps):
            draw = []
            for _ in range(len(values)):
                st = xorshift(st); draw.append(values[st % len(values)])
            est.append(median_standard(draw))
        est.sort(); return est[reps*25//1000], est[reps*975//1000]

    def boot_cluster(clusters, reps=2000):
        keys = sorted(clusters); st, est = 0x6a09e667f3bcc909, []
        for _ in range(reps):
            draw = []
            for _ in range(len(keys)):
                st = xorshift(st); grp = clusters[keys[st % len(keys)]]
                for _ in range(len(grp)):
                    st = xorshift(st); draw.append(grp[st % len(grp)])
            est.append(median_standard(draw))
        est.sort(); return est[reps*25//1000], est[reps*975//1000]

    flat_lo, flat_hi = boot_flat(per_source)
    clus_lo, clus_hi = boot_cluster(families)
    collapsed = [median_standard(v) for v in families.values()]
    coll_lo, coll_hi = boot_flat(collapsed)

    print(f"\n  {'analysis':32s} {'n':>3s} {'point':>7s} {'95% CI':>16s} {'spans parity':>13s}")
    print(f"  {'-'*32} {'-'*3} {'-'*7} {'-'*16} {'-'*13}")
    for label, n, pt, lo, hi in [
        ("source-level (as published)", len(per_source), median_standard(per_source), flat_lo, flat_hi),
        ("two-stage cluster bootstrap", len(per_source), median_standard(per_source), clus_lo, clus_hi),
        ("naive family collapse",       len(collapsed), median_standard(collapsed), coll_lo, coll_hi),
    ]:
        print(f"  {label:32s} {n:>3d} {pt:>7.2f} {f'[{lo:.2f}, {hi:.2f}]':>16s} "
              f"{str(hi >= 1.0):>13s}")

    wins = sum(1 for v in collapsed if v < 1.0)
    from math import comb, log
    p_sign = sum(comb(len(collapsed), k)
                 for k in range(wins, len(collapsed)+1)) / 2**len(collapsed)

    # Wilcoxon signed rank on the log ratios. Same distribution-free question as
    # the sign test, but it keeps the magnitudes the sign test throws away.
    # Exact null: every rank is independently added or not, so count subset sums.
    dev = sorted((log(v) for v in collapsed), key=abs)
    n = len(dev)
    w_plus = sum(i + 1 for i, d in enumerate(dev) if d > 0)
    total = n * (n + 1) // 2
    counts = [0] * (total + 1)
    counts[0] = 1
    for rank in range(1, n + 1):
        for s in range(total, rank - 1, -1):
            counts[s] += counts[s - rank]
    p_wilcoxon = min(1.0, 2 * min(sum(counts[:w_plus+1]), sum(counts[w_plus:])) / 2**n)

    print(f"\n  {wins}/{len(collapsed)} families favour the compiled plan")
    print(f"  Wilcoxon signed-rank, log ratios, two-sided : p = {p_wilcoxon:.3f}   <- reported")
    print(f"  sign test, discards magnitude, one-sided    : p = {p_sign:.3f}")

rule()
wrap("""The naive collapse is the tempting correction and it is the WRONG one: it weights
a one-column family exactly like a ten-column family, and the singleton families here
are the worst cases. The two-stage cluster bootstrap respects within-family
correlation while preserving cluster sizes, which is why the paper reports it as the
principled figure and the collapse as a pessimistic bound. Between the two
distribution-free tests the paper reports the Wilcoxon, because the sign test scores
a family at 0.04 exactly like one at 0.96; even so, neither clears 0.05.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §8 (c) — What the measurements do NOT establish. Read this before quoting
# any number from this notebook.
# ═════════════════════════════════════════════════════════════════════════════
rule("§8 — scope boundaries, stated by the paper")

limits = [
    ("Research claim",
     "an exploratory study of representation-derived facts. NOT a production engine, "
     "NOT a universal format, NOT a claim to beat Parquet."),
    ("Sampling",
     f"{claim('WitCensusSources')} census sources and {claim('WitSources')} study sources, chosen for codec "
     "diversity — not a random sample of database workloads."),
    ("Cold I/O",
     "'cold' means the OS page cache was evicted with posix_fadvise. That is advisory "
     "and page-cache-only; the device sits behind a RAID controller whose DRAM cache "
     "it cannot evict. Within-run repeats agree to ~0.1%, yet the ratio has varied by "
     "more than ten percent ACROSS runs. Precision is not reproducibility."),
    ("Aggregation deficit",
     "unexplained. Tuning the bit-unpack kernel left it unchanged; removing a "
     "per-value heap allocation cut it by about a third without closing it."),
    ("Additional plans",
     "feasibility evidence, not benchmarks. Dictionary translation is a median over "
     f"only {claim('WitDictCells')} cells, so it tracks decoder revisions sharply."),
    ("Trust model",
     "checked facts trust the encoder plus a checksummed descriptor. The checksum "
     "detects corruption, not a wrong claim. Weaker than proof-carrying code."),
    ("Out of scope",
     "joins, group-by, top-k, updates, multi-predicate planning, SIMD, "
     "multithreading, NVMe, object stores, concurrent readers."),
]
for title, body in limits:
    print(f"\n  {title}")
    wrap(body, indent="      ")

---
## §9 Conclusion

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §9 — The claim in one place, and the evidence chain that backs it.
# ═════════════════════════════════════════════════════════════════════════════
rule("§9 — conclusion")

wrap("""An encoding constrains its data, but the constraint is useful only when a query
can name it and the layout can expose the bytes. Witness separates those
responsibilities: decoder rules derive facts, facts authorise algorithms, layout rules
close the access set, and missing evidence falls back to a scan.""", indent="  ")

print()
wrap("""The results mark a boundary rather than a winner. Order evidence removes the
dominant discovery stage, yet selective access costs bytes, and fragmented reads can
erase a logical-byte advantage entirely. These are not exceptions to the idea — they
define it.""", indent="  ")

print()
wrap("""Encodings deserve evaluation not only by how compactly they store values or how
fast they decode, but by which claims they justify, and what acting on them costs.""",
     indent="  ")

rule("the evidence chain")
print(f"""  every displayed number  ->  experiments/results/claim_manifest.csv
  every manifest value    ->  a canonical CSV produced by a study binary
  every study binary      ->  frozen generated kernels + checksum-pinned inputs
  every frozen kernel     ->  an exact source fingerprint over the derivation modules

  claims in this manifest : {len(CLAIMS)}
  crate version           : {claim('WitCrateVersion')}
  source fingerprint      : {claim('WitRuleFingerprint')}

  Reproduce: ./reproduce.sh      Regenerate claims only: make claims
""")
wrap("""No reported statistic is hand-entered. If you change a measurement, the manifest
changes, the paper's macros change, and a regression test fails if the prose does
not.""", indent="  ")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# APPENDIX — self-check. Confirms this notebook read the real artifact rather
# than its fallbacks, so a reader knows which mode they are in.
# ═════════════════════════════════════════════════════════════════════════════
rule("APPENDIX — provenance of everything above")

expected = [
    "claim_manifest.csv",
    "invariant_census/summary.csv",
    "invariant_census/sources.csv",
    "certificate_study/summary.csv",
    "predicate_pipeline/source_summary.csv",
    "predicate_pipeline/certificate_summary.csv",
    "real_access/storage_scan.csv",
]
missing = [p for p in expected if not (RESULTS / p).is_file()]
for p in expected:
    print(f"  {'OK  ' if (RESULTS / p).is_file() else 'MISS'}  {p}")

print()
if missing:
    print(f"  MODE: PARTIAL — {len(missing)} source(s) missing; those cells used DUMMY data.")
    print("        Numbers from those cells are illustrative and are NOT the paper's.")
else:
    print("  MODE: LIVE — every measured number above came from the artifact's own CSVs.")
    print("        They are the same values the paper's macros are generated from.")

print(f"\n  Toy/synthetic data was used ONLY in: §1, §2, §3 (marked DUMMY in-cell),")
print(f"  and the CountRuns illustration in §6. Those are pedagogical by design.")